# BitcoinGraphGuard — Phase 7: Heterogeneous Graph Transformer (HGT)

**Phase:** 7 of 11 · Heterogeneous Graph Transformer (HGT)
**Environment:** Google Colab / Kaggle (CUDA GPU strongly recommended).
**Inputs:** Raw Elliptic++ CSVs (`txs_features.csv`, `txs_classes.csv`, `txs_edgelist.csv`, `wallets_features.csv`, `wallets_classes.csv`, `AddrTx_edgelist.csv`, `TxAddr_edgelist.csv`, `AddrAddr_edgelist.csv`).

---

## 1. Executive Summary & Research Context

Across Phases 1 through 6, BitcoinGraphGuard established a rigorous empirical benchmark on the Elliptic++ dataset:
$$\text{Prevalence (0.0650)} \to \text{Logistic Regression (0.2917)} \to \text{MLP (0.4768)} \to \text{HeteroRGCN (0.4682)} \to \text{GraphSAGE (0.6209)} \to \mathbf{\text{XGBoost (0.8013)}}$$

### The Core Scientific Puzzle
1. **The Performance Gap on Test 35–49**:
   - **XGBoost (0.8013 PR-AUC)** dominates tabular evaluation by non-linearly partitioning 72 pre-aggregated neighborhood features.
   - **HeteroRGCN (0.4682 PR-AUC)** and **the GraphSAGE v2 best variant, O4 (0.6001 PR-AUC)**, underperformed XGBoost significantly.
2. **The Late Temporal Collapse on Steps 43–49**:
   - In time steps 43–49, illicit transaction prevalence abruptly collapses from 9.16% (steps 35–42) to **2.53%**, accompanied by severe covariate drift (adversarial validation AUC 1.0000 on `Aggregate_feature_*`).
   - Every prior model collapsed to $\approx 0.04 - 0.055$ PR-AUC on steps 43–49 (XGBoost 0.0427, GraphSAGE 0.0505, RGCN 0.0550).
3. **The Dilution Hypothesis (Phase 4 & Phase 6 Explainability)**:
   - Captum and test-time relation ablation revealed that HeteroRGCN over-relies on recurrent address pathways (`addr_to_tx`, `tx_to_addr`, `addr_to_addr`).
   - Because RGCN uses **uniform mean aggregation** ($\frac{1}{|\mathcal{N}_i^r|} \sum W_r h_j$), dense wallet-to-wallet transfers (`AddrAddr` accounts for 65% of all edges) and high-degree exchange hot wallets drown out subtle illicit signals.

---

## 2. Phase 7 Objectives & Hypotheses

This notebook implements the **Heterogeneous Graph Transformer (HGT)** (Hu et al., *WWW 2020*) to test two central questions:
1. **Hypothesis 1 (Primary Benchmark 35–49)**: Can dynamic, learned relation-specific and source/target-dependent attention gating close the gap between GNNs and XGBoost (0.8013)?
2. **Hypothesis 2 (Late Drift Window 43–49 — The Critical Test)**: Can learned attention prevent or alleviate the collapse to $\sim 0.05$ on steps 43–49 by selectively filtering out noisy background exchange transactions?
3. **Direct Attention Extraction on True Positives**: We extract edge-level attention weights for `AddrAddr` and `AddrTx` on test True Positives to directly verify whether HGT learns to downweight high-degree reused addresses.
4. **Definitive Diagnosis**: If HGT still collapses to $\sim 0.05$ on 43–49, we establish whether this failure is an architectural aggregation problem or an **irreducible regime shift** (unseen entities and missing supervision), providing a clear recommendation for Phase 8 MLOps.

In [ ]:
import gc
import json
import os
import platform
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.sparse import coo_matrix
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

try:
    import torch_geometric
except ImportError:
    !pip install -q torch-geometric
    import torch_geometric

from torch_geometric.data import HeteroData
from torch_geometric.loader import HeteroNeighborLoader
from torch_geometric.nn import HGTConv, Linear

SEED = 42
N_STEPS = 49

# Strict Chronological Partitioning (Identical to Phases 1-6)
FIT_MIN, FIT_MAX = 1, 24
VAL_MIN, VAL_MAX = 25, 34
TRAIN_MIN, TRAIN_MAX = 1, 34
TEST_MIN, TEST_MAX = 35, 49
TEST_EARLY_MAX = 42
DRIFT_MIN = 43

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print(f"python {sys.version.split()[0]} | platform {platform.platform()}")
print(f"numpy {np.__version__} | pandas {pd.__version__} | torch {torch.__version__} | PyG {torch_geometric.__version__}")
print(f"Hardware Acceleration Device: {device}")


In [ ]:
FILES = {
    "txs_features": "txs_features.csv",
    "txs_classes": "txs_classes.csv",
    "txs_edgelist": "txs_edgelist.csv",
    "wallets_features": "wallets_features.csv",
    "wallets_classes": "wallets_classes.csv",
    "addr_tx": "AddrTx_edgelist.csv",
    "tx_addr": "TxAddr_edgelist.csv",
    "addr_addr": "AddrAddr_edgelist.csv",
}

DRIVE_ROOT = Path(os.environ.get("BITCOINGUARD_DRIVE_ROOT", "/content/drive/MyDrive/Projects/Bitcoin Graph"))

def resolve_data_dir() -> Path:
    override = os.environ.get("ELLIPTIC_DATA_DIR")
    candidates = [
        Path(override) if override else None,
        DRIVE_ROOT / "Dataset",
        Path("/content/drive/MyDrive/elliptic-plus-plus"),
        Path("/content/elliptic-plus-plus"),
        Path("/kaggle/input/elliptic-plus-plus"),
        Path("/kaggle/input/elliptic"),
        Path.cwd() / "Og data",
        Path.cwd().parent / "Og data",
    ]
    for candidate in candidates:
        if candidate and candidate.is_dir() and (candidate / FILES["txs_features"]).exists():
            return candidate
    raise FileNotFoundError("Elliptic++ dataset directory not found. Please set ELLIPTIC_DATA_DIR.")

DATA_DIR = resolve_data_dir()
OUT_DIR = Path("results/hgt")
FIG_DIR = OUT_DIR / "figures"
for _dir in (OUT_DIR, FIG_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

CHECKS = []

def check(name: str, value, expected=None, note: str = ""):
    ok = (value == expected) if expected is not None else True
    CHECKS.append({"check": name, "value": value, "expected": expected, "ok": bool(ok), "note": note})
    flag = "ok" if ok else "FAIL"
    print(f"[{flag}] {name}: {value}" + (f" (expected {expected})" if expected is not None else ""))

def save_table(name: str, frame: pd.DataFrame):
    frame.to_csv(OUT_DIR / name, index=False)
    print(f"Saved: {OUT_DIR / name}")

def save_fig(name: str):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {path}")

def save_json(name: str, payload: dict):
    with open(OUT_DIR / name, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(f"Saved JSON: {OUT_DIR / name}")

def tick(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

print(f"Dataset root : {DATA_DIR}")
print(f"Artifact dir : {OUT_DIR}")


## 3. Data Loading & Heterogeneous Graph Assembly

We load the complete Elliptic++ multi-relational dataset:
- **Transaction Nodes (`tx`)**: 203,769 transactions with 165 non-domain features (93 Local + 72 Aggregate).
- **Address Nodes (`addr`)**: 822,942 wallet addresses with 55 behavioral features.
- **4 Directed Edge Relations**:
  1. `('tx', 'tx_to_tx', 'tx')`: 234,355 directed edges (100% intra-step).
  2. `('addr', 'addr_to_tx', 'tx')`: 477,117 wallet-to-transaction funding edges (`AddrTx`).
  3. `('tx', 'tx_to_addr', 'addr')`: 837,124 transaction-to-wallet deposit edges (`TxAddr`).
  4. `('addr', 'addr_to_addr', 'addr')`: 2,868,964 direct wallet transfers (`AddrAddr`).

**Strict Temporal Availability & Deduplication**:
- Wallet snapshots are deduplicated to unique `(address, Time step)` pairs.
- First-seen time steps are calculated across all snapshots and incident transaction edges.
- Address features for any time step $t$ are constructed strictly using snapshots with $t_{\text{snapshot}} \le t$ (zero future lookahead).

In [ ]:
tick("Loading raw Elliptic++ CSVs...")

def read_header(path: Path):
    return pd.read_csv(path, nrows=0).columns.tolist()

txs_columns = read_header(DATA_DIR / FILES["txs_features"])
all_tx_features = [c for c in txs_columns if c not in ("txId", "Time step")]
domain_columns = [c for c in all_tx_features if not c.startswith(("Local_feature", "Aggregate_feature"))]
tx_model_features = [c for c in all_tx_features if c not in domain_columns]

txs_classes = pd.read_csv(DATA_DIR / FILES["txs_classes"], dtype={"txId": str})
txs = pd.read_csv(
    DATA_DIR / FILES["txs_features"],
    usecols=["txId", "Time step"] + tx_model_features,
    dtype={"txId": str, "Time step": "int16", **{c: "float32" for c in tx_model_features}},
)

# Binary label mapping: 1 -> 1 (illicit), 2 -> 0 (licit), 3/unknown -> 2 (unlabeled)
label_map = {1: 1, 2: 0, 3: 2, '1': 1, '2': 0, '3': 2, 'unknown': 2}
txs["class"] = txs["txId"].map(txs_classes.set_index("txId")["class"]).map(label_map).fillna(2).astype("int8")
check("data: tx model feature count", len(tx_model_features), 165)
check("data: total transaction count", len(txs), 203769)

wallets_classes = pd.read_csv(DATA_DIR / FILES["wallets_classes"], dtype={"address": str})
wallet_cols = read_header(DATA_DIR / FILES["wallets_features"])
wallet_feature_cols = [c for c in wallet_cols if c not in ("address", "Time step")]

wallets_raw_df = pd.read_csv(
    DATA_DIR / FILES["wallets_features"],
    dtype={"address": str, "Time step": "int16", **{c: "float32" for c in wallet_feature_cols}},
)
wallets_df = wallets_raw_df.drop_duplicates(subset=["address", "Time step"]).reset_index(drop=True)
del wallets_raw_df
gc.collect()

tx_id_to_idx = {tx_id: idx for idx, tx_id in enumerate(txs["txId"].values)}
addr_to_idx = {addr: idx for idx, addr in enumerate(wallets_classes["address"].values)}
N_TX = len(txs)
N_ADDR = len(wallets_classes)

edges_tx_tx = pd.read_csv(DATA_DIR / FILES["txs_edgelist"], dtype=str)
edges_addr_tx = pd.read_csv(DATA_DIR / FILES["addr_tx"], dtype=str)
edges_tx_addr = pd.read_csv(DATA_DIR / FILES["tx_addr"], dtype=str)
edges_addr_addr = pd.read_csv(DATA_DIR / FILES["addr_addr"], dtype=str)

src_tx_tx = edges_tx_tx["txId1"].map(tx_id_to_idx).values.astype(np.int64)
dst_tx_tx = edges_tx_tx["txId2"].map(tx_id_to_idx).values.astype(np.int64)
src_addr_tx = edges_addr_tx["input_address"].map(addr_to_idx).values.astype(np.int64)
dst_addr_tx = edges_addr_tx["txId"].map(tx_id_to_idx).values.astype(np.int64)
src_tx_addr = edges_tx_addr["txId"].map(tx_id_to_idx).values.astype(np.int64)
dst_tx_addr = edges_tx_addr["output_address"].map(addr_to_idx).values.astype(np.int64)
src_addr_addr = edges_addr_addr["input_address"].map(addr_to_idx).values.astype(np.int64)
dst_addr_addr = edges_addr_addr["output_address"].map(addr_to_idx).values.astype(np.int64)

# Node-id integrity: a failed map would silently cast NaN to INT64_MIN and corrupt the graph.
check("mappings: all transaction endpoints resolve to a node index", int(max(src_tx_tx.max(), dst_tx_tx.max()) < N_TX), True)
check(
    "mappings: all address endpoints resolve to a node index",
    int(max(src_addr_tx.max(), dst_addr_tx.max(), src_tx_addr.max(), dst_tx_addr.max(), src_addr_addr.max(), dst_addr_addr.max()) < N_ADDR),
    True,
)
check("mappings: all edge indices are non-negative", int(min(src_tx_tx.min(), dst_tx_tx.min(), src_addr_tx.min(), dst_addr_tx.min(), src_tx_addr.min(), dst_tx_addr.min(), src_addr_addr.min(), dst_addr_addr.min()) >= 0), True)

# Temporal availability metadata
tx_step = txs["Time step"].values.astype(np.int16)
tx_class = txs["class"].values.astype(np.int8)
is_labeled = np.isin(tx_class, [0, 1])
tx_ids = txs["txId"].values
raw_tx_features = txs[tx_model_features].to_numpy(dtype=np.float32)

wallets_df["addr_idx"] = wallets_df["address"].map(addr_to_idx)
wallets_arr = wallets_df[wallet_feature_cols].to_numpy(dtype=np.float32)
wallets_step = wallets_df["Time step"].values.astype(np.int16)
wallets_addr_idx = wallets_df["addr_idx"].values.astype(np.int64)

UNSEEN_STEP = np.int16(N_STEPS + 1)
first_seen_addr = np.full(N_ADDR, UNSEEN_STEP, dtype=np.int16)
np.minimum.at(first_seen_addr, wallets_addr_idx, wallets_step)
np.minimum.at(first_seen_addr, src_addr_tx, tx_step[dst_addr_tx])
np.minimum.at(first_seen_addr, dst_tx_addr, tx_step[src_tx_addr])

EDGE_KEYS = ["tx_to_tx", "addr_to_tx", "tx_to_addr", "addr_to_addr"]
edge_endpoints = {
    "tx_to_tx": (src_tx_tx, dst_tx_tx),
    "addr_to_tx": (src_addr_tx, dst_addr_tx),
    "tx_to_addr": (src_tx_addr, dst_tx_addr),
    "addr_to_addr": (src_addr_addr, dst_addr_addr),
}
edge_avail = {
    "tx_to_tx": tx_step[src_tx_tx],
    "addr_to_tx": tx_step[dst_addr_tx],
    "tx_to_addr": tx_step[src_tx_addr],
    "addr_to_addr": np.maximum(first_seen_addr[src_addr_addr], first_seen_addr[dst_addr_addr]),
}

# Incidence matrix for inductive tracking (seen vs unseen address context)
inc_rows = np.concatenate([dst_addr_tx, src_tx_addr])
inc_cols = np.concatenate([src_addr_tx, dst_tx_addr])
INCIDENCE = coo_matrix(
    (np.ones(inc_rows.size, dtype=np.float32), (inc_rows, inc_cols)), shape=(N_TX, N_ADDR)
).tocsr()
INCIDENCE.data[:] = 1.0

row_idx = np.repeat(np.arange(N_TX), np.diff(INCIDENCE.indptr))
addr_of_tx = INCIDENCE.indices
tx_min_addr_firstseen = np.full(N_TX, UNSEEN_STEP, dtype=np.int16)
np.minimum.at(tx_min_addr_firstseen, row_idx, first_seen_addr[addr_of_tx])
has_any_addr = np.diff(INCIDENCE.indptr) > 0

tick("Graph assembly and incidence structures complete.")


## 4. Leakage-Safe Feature Scaling & Graph Construction

To guarantee zero temporal data leakage:
1. **Target Filtering**: Unlabeled background transactions (class 3) are strictly excluded from loss computation and validation/test evaluation.
2. **Feature Scalers**: `StandardScaler` for transactions and addresses are fitted **strictly on historical training nodes** ($t \le 24$ for selection model, $t \le 34$ for refit model).
3. **Address Snapshot Aggregation**: Address feature matrices are computed using mean aggregation over historical appearances strictly up to the cutoff step ($t_{\text{snapshot}} \le \text{cutoff}$). Unseen addresses receive the historical training mean.
4. **Edge Temporal Filtering**: Edges are included only if their participating nodes/transactions are available at or before the cutoff step.

In [ ]:
def edges_up_to(cutoff: int) -> dict:
    out = {}
    for key in EDGE_KEYS:
        keep = np.nonzero(edge_avail[key] <= cutoff)[0]
        out[key] = torch.from_numpy(np.stack([edge_endpoints[key][0][keep], edge_endpoints[key][1][keep]], axis=0).astype(np.int64))
    return out

def raw_address_matrix(cutoff: int) -> np.ndarray:
    hist = wallets_step <= cutoff
    hist_addrs = wallets_addr_idx[hist]
    hist_feats = wallets_arr[hist]
    sums = np.zeros((N_ADDR, hist_feats.shape[1]), dtype=np.float32)
    counts = np.zeros(N_ADDR, dtype=np.float32)
    np.add.at(sums, hist_addrs, hist_feats)
    np.add.at(counts, hist_addrs, 1.0)
    valid = counts > 0
    sums[valid] /= counts[valid, None]
    if hist_feats.shape[0] > 0:
        global_mean = hist_feats.mean(axis=0)
        sums[~valid] = global_mean
    return sums

def build_heterodata(cutoff: int, mask_mode: str = "selection") -> tuple:
    '''Construct PyG HeteroData with strict leakage-safe feature scaling.'''
    data = HeteroData()

    # 1. Scaler fitting on training subset
    fit_tx_mask = is_labeled & (tx_step >= FIT_MIN) & (tx_step <= FIT_MAX) if mask_mode == "selection" else is_labeled & (tx_step >= TRAIN_MIN) & (tx_step <= TRAIN_MAX)
    scaler_tx = StandardScaler().fit(raw_tx_features[fit_tx_mask])
    data['tx'].x = torch.from_numpy(scaler_tx.transform(raw_tx_features).astype(np.float32))

    raw_addr = raw_address_matrix(cutoff)
    fit_addr_mask = first_seen_addr <= (FIT_MAX if mask_mode == "selection" else TRAIN_MAX)
    if fit_addr_mask.sum() > 0:
        scaler_addr = StandardScaler().fit(raw_addr[fit_addr_mask])
        data['addr'].x = torch.from_numpy(scaler_addr.transform(raw_addr).astype(np.float32))
    else:
        data['addr'].x = torch.from_numpy(raw_addr.astype(np.float32))

    # 2. Target labels (for transactions only)
    data['tx'].y = torch.tensor(tx_class, dtype=torch.float32)

    # 3. Directed Multi-relational Edges
    edge_dict = edges_up_to(cutoff)
    data['tx', 'tx_to_tx', 'tx'].edge_index = edge_dict["tx_to_tx"]
    data['addr', 'addr_to_tx', 'tx'].edge_index = edge_dict["addr_to_tx"]
    data['tx', 'tx_to_addr', 'addr'].edge_index = edge_dict["tx_to_addr"]
    data['addr', 'addr_to_addr', 'addr'].edge_index = edge_dict["addr_to_addr"]

    # 4. Partition Masks
    if mask_mode == "selection":
        data['tx'].train_mask = torch.from_numpy(is_labeled & (tx_step >= FIT_MIN) & (tx_step <= FIT_MAX))
    else:
        data['tx'].train_mask = torch.from_numpy(is_labeled & (tx_step >= TRAIN_MIN) & (tx_step <= TRAIN_MAX))
    data['tx'].val_mask = torch.from_numpy(is_labeled & (tx_step >= VAL_MIN) & (tx_step <= VAL_MAX))
    data['tx'].test_mask = torch.from_numpy(is_labeled & (tx_step >= TEST_MIN) & (tx_step <= TEST_MAX))

    return data

tick("Building static HeteroData for Selection (1-24 fit / 25-34 val) and Refit (1-34 train)...")
# Selection uses a step-24 graph so that neither the fit nor the validation pass can
# observe any edge from the future; refit additionally admits the step 25-34 edges
# because those steps are part of the final training window.
selection_data = build_heterodata(FIT_MAX, mask_mode="selection")
refit_data = build_heterodata(TRAIN_MAX, mask_mode="refit")

check("graph: tx node count", selection_data['tx'].num_nodes, 203769)
check("graph: addr node count", selection_data['addr'].num_nodes, 822942)
check("graph: selection train nodes", int(selection_data['tx'].train_mask.sum()), 29894)
check("graph: selection val nodes", int(selection_data['tx'].val_mask.sum()), 9091)
check("graph: test nodes", int(refit_data['tx'].test_mask.sum()), 16670)


## 4b. Leakage Audit

Before any model is trained we assert the leakage controls that every prior phase enforced, so a
result cannot be explained away as lookahead:

1. **Disjoint step sets** — fit (1–24), validation (25–34) and test (35–49) share no time step.
2. **No future edges** — the selection graph is built from a step-24 cutoff and the refit graph from
   a step-34 cutoff, so no message can traverse an edge that did not yet exist at training time.
3. **Unknown is not licit** — class 3 is excluded from every supervision and evaluation mask.
4. **Training-only statistics** — feature scalers and `pos_weight` are derived from labelled
   training steps only, and the operating threshold is fit on 25–34 and frozen before test.
5. **Carry-forward guardrail** — an address link can never mark a positive, so `has_addresses` is a
   one-sided licit indicator rather than evidence of fraud.

In [ ]:
claim_steps = {
    "fit": set(range(FIT_MIN, FIT_MAX + 1)),
    "validation": set(range(VAL_MIN, VAL_MAX + 1)),
    "train": set(range(TRAIN_MIN, TRAIN_MAX + 1)),
    "test": set(range(TEST_MIN, TEST_MAX + 1)),
}
check("leakage: fit and validation step sets disjoint", len(claim_steps["fit"] & claim_steps["validation"]), 0)
check("leakage: fit and test step sets disjoint", len(claim_steps["fit"] & claim_steps["test"]), 0)
check("leakage: train(1-34) and test(35-49) step sets disjoint", len(claim_steps["train"] & claim_steps["test"]), 0)

# The selection graph must contain exactly the edges that were available by step 24;
# the refit graph must contain exactly the edges available by step 34.
fit_counts = {key: int((edge_avail[key] <= FIT_MAX).sum()) for key in EDGE_KEYS}
kept_counts = {key: int((edge_avail[key] <= TRAIN_MAX).sum()) for key in EDGE_KEYS}
for edge_type, key in zip(
    [("tx", "tx_to_tx", "tx"), ("addr", "addr_to_tx", "tx"), ("tx", "tx_to_addr", "addr"), ("addr", "addr_to_addr", "addr")],
    EDGE_KEYS,
):
    check(f"leakage: selection graph '{key}' keeps only step<=24 edges", int(selection_data[edge_type].edge_index.size(1)), fit_counts[key])
    check(f"leakage: refit graph '{key}' keeps only step<=34 edges", int(refit_data[edge_type].edge_index.size(1)), kept_counts[key])

# Unknown (class 3) must never enter a supervision or evaluation mask.
supervised = (selection_data["tx"].train_mask | selection_data["tx"].val_mask | refit_data["tx"].test_mask).cpu().numpy()
check("leakage: no unlabeled (class 3) node in any mask", int((tx_class[supervised] == 2).sum()), 0)
check("leakage: selection scaler fit window is labelled 1-24 only", int((is_labeled & (tx_step >= FIT_MIN) & (tx_step <= FIT_MAX)).sum()), 29894)
check("leakage: refit scaler fit window is labelled 1-34 only", int((is_labeled & (tx_step >= TRAIN_MIN) & (tx_step <= TRAIN_MAX)).sum()), 38985)

# Carry-forward guardrail from Phases 2/2b: every illicit transaction has an address link.
check("guardrail: has_addresses never marks a positive", int((~has_any_addr & (tx_class == 1)).sum()), 0)
tick("Leakage audit assertions recorded.")


## 5. Heterogeneous Graph Transformer (HGT) Architecture

### 5.1. Mathematical Formulation (Hu et al., 2020)
For a source node $s$ of type $\tau(s)$ and a target node $t$ of type $\tau(t)$ connected by relation $e = (s, \phi(e), t)$:

1. **Heterogeneous Mutual Attention**:
   $$\text{Attention}(s, e, t) = \text{Softmax}_{j \in \mathcal{N}(t)}\left( \bigoplus_{h=1}^H \frac{K^h(s) W_{\phi(e)}^{\text{ATT}} {Q^h(t)}^T}{\sqrt{d}} \cdot \mu_{\langle \tau(s), \phi(e), \tau(t) \rangle} \right)$$
   Where:
   - $Q^h(t) = \text{Linear}_{\tau(t)}^Q(h_t)$ and $K^h(s) = \text{Linear}_{\tau(s)}^K(h_s)$.
   - $W_{\phi(e)}^{\text{ATT}}$ is the relation-specific attention projection matrix.
   - $\mu_{\langle \tau(s), \phi(e), \tau(t) \rangle}$ is a learned relational scaling scalar (`a_rel`).
2. **Heterogeneous Message Passing**:
   $$\text{Message}(s, e, t) = \bigoplus_{h=1}^H \left( \text{Linear}_{\tau(s)}^V(h_s) W_{\phi(e)}^{\text{MSG}} \right)$$
3. **Target-Specific Aggregation**:
   $$\tilde{h}_t = \sum_{s \in \mathcal{N}(t)} \text{Attention}(s, e, t) \cdot \text{Message}(s, e, t)$$
   $$h_t^{(l+1)} = \text{GELU}\left( \text{Linear}_{\tau(t)}^{\text{up}}(\tilde{h}_t) + h_t^{(l)} \right)$$

### 5.2. Layer Specifications
- **Input Projections**: Linear transforms for `tx` ($165 \to 64$) and `addr` ($55 \to 64$).
- **Convolutions**: 2 layers of `HGTConv` with $H = 4$ attention heads, hidden dimension $d = 64$ ($d_k = 16$ per head).
- **Group Aggregation**: `'sum'` across incoming relation heads.
- **Regularization**: Dropout $p = 0.3$, GELU non-linear activations.
- **Classifier**: Linear layer mapping transaction representation ($64 \to 1$ logit).

In [ ]:
class HGT(nn.Module):
    '''Heterogeneous Graph Transformer for Bitcoin fraud detection.'''
    def __init__(
        self,
        hidden_channels: int,
        out_channels: int,
        num_heads: int,
        num_layers: int,
        data_metadata: tuple,
        tx_in: int = 165,
        addr_in: int = 55,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.dropout = dropout

        # Input feature projections for heterogeneous node types
        self.lin_dict = nn.ModuleDict({
            'tx': Linear(tx_in, hidden_channels),
            'addr': Linear(addr_in, hidden_channels),
        })

        # Multi-layer HGT message passing
        self.convs = nn.ModuleList([
            HGTConv(
                in_channels=hidden_channels,
                out_channels=hidden_channels,
                metadata=data_metadata,
                heads=num_heads,
                group='sum',
            )
            for _ in range(num_layers)
        ])

        # Final transaction classification head
        self.lin_out = Linear(hidden_channels, out_channels)

    def forward(self, x_dict: dict, edge_index_dict: dict) -> torch.Tensor:
        # Initial projection with GELU
        h_dict = {nt: F.gelu(self.lin_dict[nt](x)) for nt, x in x_dict.items()}

        for conv in self.convs:
            h_dict = conv(h_dict, edge_index_dict)
            h_dict = {
                nt: F.gelu(F.dropout(x, p=self.dropout, training=self.training))
                for nt, x in h_dict.items()
            }

        # Output logits for transaction nodes
        return self.lin_out(h_dict['tx'])


def _node_linear(container, node_type: str):
    '''Return the per-node-type Linear from a ModuleDict/ParameterDict, or a shared Linear.'''
    if container is None:
        raise AttributeError("HGTConv exposes no Q/K projection for this node type (unexpected PyG version).")
    if isinstance(container, nn.Linear):
        return container
    try:
        return container[node_type]
    except (KeyError, TypeError):
        for name, module in container.named_modules():
            if isinstance(module, nn.Linear) and node_type in name:
                return module
        raise


def _relation_scalar(conv, edge_type: tuple, device) -> torch.Tensor:
    '''Look up the learned HGT relation scaling a_rel, tolerating PyG key conventions.'''
    a_rel = getattr(conv, "a_rel", None)
    if a_rel is not None:
        src_type, rel_name, dst_type = edge_type
        for key in (edge_type, "__".join(edge_type), f"{src_type}_{rel_name}_{dst_type}", rel_name):
            try:
                if key in a_rel:
                    return a_rel[key].to(device)
            except TypeError:
                continue
    return torch.ones(int(getattr(conv, "heads", 4)), device=device)


def compute_hgt_edge_attention(
    model: HGT,
    x_dict: dict,
    edge_index_dict: dict,
    edge_type: tuple,
    layer_idx: int = 0,
) -> tuple:
    '''Extract softmax-normalised edge attention weights for one relation in an HGT layer.

    Attention follows Hu et al. (2020): per-head logits ``(Q_dst . K_src) * a_rel / sqrt(d_k)``
    are normalised over each destination node's incoming neighbours and averaged over heads.
    Returns ``(attention_weights, attention_logits, edge_index)``.
    '''
    from torch_geometric.utils import softmax as pyg_softmax

    model.eval()
    conv = model.convs[layer_idx]
    src_type, rel_name, dst_type = edge_type
    edge_index = edge_index_dict[edge_type]
    if edge_index.size(1) == 0:
        return np.array([]), np.array([]), np.empty((2, 0), dtype=np.int64)

    with torch.no_grad():
        if layer_idx == 0:
            h_src = F.gelu(model.lin_dict[src_type](x_dict[src_type]))
            h_dst = F.gelu(model.lin_dict[dst_type](x_dict[dst_type]))
        else:
            h_dict = {nt: F.gelu(model.lin_dict[nt](x)) for nt, x in x_dict.items()}
            h_dict = model.convs[0](h_dict, edge_index_dict)
            h_src = F.gelu(h_dict[src_type])
            h_dst = F.gelu(h_dict[dst_type])

        heads = int(getattr(conv, "heads", 4))
        q_proj = _node_linear(getattr(conv, "q_lin", None) or getattr(conv, "lin_q", None), dst_type)
        k_proj = _node_linear(getattr(conv, "k_lin", None) or getattr(conv, "lin_k", None), src_type)
        d_k = q_proj.out_features // heads
        q = q_proj(h_dst).view(-1, heads, d_k)
        k = k_proj(h_src).view(-1, heads, d_k)

        src_idx, dst_idx = edge_index[0], edge_index[1]
        q_dst = q[dst_idx]  # [num_edges, heads, d_k]
        k_src = k[src_idx]  # [num_edges, heads, d_k]

        a_scalar = _relation_scalar(conv, edge_type, q.device)
        logits = (q_dst * k_src).sum(dim=-1) * a_scalar / (d_k ** 0.5)  # [E, heads]
        # Softmax over each destination node's incoming edges, independently per head.
        weights = pyg_softmax(logits, dst_idx, num_nodes=q.size(0))

        mean_weights = weights.mean(dim=-1).cpu().numpy()
        mean_logits = logits.mean(dim=-1).cpu().numpy()
        return mean_weights, mean_logits, edge_index.cpu().numpy()

print("HGT architecture and attention extraction helper defined.")


In [ ]:
def train_hgt(
    data: HeteroData,
    epochs: int = 80,
    lr: float = 0.001,
    weight_decay: float = 1e-4,
    dropout: float = 0.3,
    batch_size: int = 1024,
    num_neighbors: list = None,
    seed: int = SEED,
) -> tuple:
    '''Train HGT with HeteroNeighborLoader mini-batching and validation early stopping.'''
    if num_neighbors is None:
        num_neighbors = [10, 10]

    torch.manual_seed(seed)
    model = HGT(
        hidden_channels=64,
        out_channels=1,
        num_heads=4,
        num_layers=2,
        data_metadata=data.metadata(),
        tx_in=165,
        addr_in=55,
        dropout=dropout,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Class-weighted loss computed strictly on training mask
    train_labels = data['tx'].y[data['tx'].train_mask]
    positives = train_labels.sum().item()
    negatives = (train_labels == 0).sum().item()
    pos_weight = torch.tensor([negatives / max(1, positives)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    train_loader = HeteroNeighborLoader(
        data,
        num_neighbors=num_neighbors,
        input_nodes=('tx', data['tx'].train_mask),
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = HeteroNeighborLoader(
        data,
        num_neighbors=num_neighbors,
        input_nodes=('tx', data['tx'].val_mask),
        batch_size=batch_size,
        shuffle=False,
    )

    best_val_pr_auc = -1.0
    best_epoch = 1
    best_weights = None
    history = {"epoch": [], "train_loss": [], "val_pr_auc": []}

    print(f"Training HGT: pos_weight={pos_weight.item():.4f}, batch_size={batch_size}, neighbors={num_neighbors}")

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total_nodes = 0.0, 0

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch.x_dict, batch.edge_index_dict)
            bs = batch['tx'].batch_size
            target_logits = logits[:bs].squeeze(1) if logits[:bs].dim() > 1 else logits[:bs].squeeze(0)
            target_y = batch['tx'].y[:bs]
            loss = criterion(target_logits, target_y)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * bs
            total_nodes += bs

        train_loss = total_loss / max(1, total_nodes)
        history["train_loss"].append(train_loss)

        # Validation evaluation
        model.eval()
        val_probs, val_y = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                logits = model(batch.x_dict, batch.edge_index_dict)
                bs = batch['tx'].batch_size
                probs = torch.sigmoid(logits[:bs].squeeze(1) if logits[:bs].dim() > 1 else logits[:bs].squeeze(0)).cpu().numpy()
                y = batch['tx'].y[:bs].cpu().numpy()
                val_probs.extend(probs.tolist())
                val_y.extend(y.tolist())

        val_pr_auc = float(average_precision_score(val_y, val_probs))
        history["val_pr_auc"].append(val_pr_auc)
        history["epoch"].append(epoch)

        if val_pr_auc > best_val_pr_auc:
            best_val_pr_auc = val_pr_auc
            best_epoch = epoch
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0 or epoch == epochs:
            print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {train_loss:.4f} | Val PR-AUC: {val_pr_auc:.4f} (Best: {best_val_pr_auc:.4f} @ Ep {best_epoch})")

    if best_weights is not None:
        model.load_state_dict(best_weights)

    return model, best_epoch, best_val_pr_auc, pd.DataFrame(history), pos_weight


def refit_hgt(
    data: HeteroData,
    epochs: int,
    lr: float = 0.001,
    weight_decay: float = 1e-4,
    dropout: float = 0.3,
    batch_size: int = 1024,
    num_neighbors: list = None,
    seed: int = SEED,
) -> tuple:
    '''Refit HGT on full historical training set (1-34) for fixed epoch count.'''
    if num_neighbors is None:
        num_neighbors = [10, 10]

    torch.manual_seed(seed)
    model = HGT(
        hidden_channels=64,
        out_channels=1,
        num_heads=4,
        num_layers=2,
        data_metadata=data.metadata(),
        tx_in=165,
        addr_in=55,
        dropout=dropout,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_labels = data['tx'].y[data['tx'].train_mask]
    positives = train_labels.sum().item()
    negatives = (train_labels == 0).sum().item()
    pos_weight = torch.tensor([negatives / max(1, positives)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    train_loader = HeteroNeighborLoader(
        data,
        num_neighbors=num_neighbors,
        input_nodes=('tx', data['tx'].train_mask),
        batch_size=batch_size,
        shuffle=True,
    )

    print(f"Refitting HGT on Steps 1-34 for {epochs} epochs (pos_weight={pos_weight.item():.4f})...")
    model.train()
    for epoch in range(1, epochs + 1):
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch.x_dict, batch.edge_index_dict)
            bs = batch['tx'].batch_size
            target_logits = logits[:bs].squeeze(1) if logits[:bs].dim() > 1 else logits[:bs].squeeze(0)
            target_y = batch['tx'].y[:bs]
            loss = criterion(target_logits, target_y)
            loss.backward()
            optimizer.step()

    return model, pos_weight


@torch.no_grad()
def infer_hgt(model: nn.Module, data: HeteroData, mask: torch.Tensor, batch_size: int = 1024, num_neighbors: list = None) -> np.ndarray:
    '''Generate test predictions using HeteroNeighborLoader.'''
    if num_neighbors is None:
        num_neighbors = [10, 10]

    model.eval()
    loader = HeteroNeighborLoader(
        data,
        num_neighbors=num_neighbors,
        input_nodes=('tx', mask),
        batch_size=batch_size,
        shuffle=False,
    )
    all_probs = []
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x_dict, batch.edge_index_dict)
        bs = batch['tx'].batch_size
        probs = torch.sigmoid(logits[:bs].squeeze(1) if logits[:bs].dim() > 1 else logits[:bs].squeeze(0)).cpu().numpy()
        all_probs.extend(probs.tolist())
    return np.array(all_probs, dtype=np.float64)


In [ ]:
# Hyperparameters
EPOCHS = 80
BATCH_SIZE = 1024
NUM_NEIGHBORS = [10, 10]
LR = 0.001
WD = 1e-4
DROPOUT = 0.3

tick("--- Phase 1: Training HGT Selection Model (Fit 1-24 / Val 25-34) ---")
start_time = time.time()
hgt_selection_model, best_epoch, val_pr_auc_best, history_df, pos_weight_fit = train_hgt(
    selection_data,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WD,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    num_neighbors=NUM_NEIGHBORS,
)
fit_duration = time.time() - start_time
print(f"Selection Model Complete: Best Epoch {best_epoch} | Validation PR-AUC {val_pr_auc_best:.4f} in {fit_duration:.1f}s")
save_table("hgt_training_history.csv", history_df)

tick("--- Phase 2: Refitting HGT Model on Full Historical Window (Train 1-34) ---")
start_time = time.time()
hgt_refit_model, pos_weight_train = refit_hgt(
    refit_data,
    epochs=best_epoch,
    lr=LR,
    weight_decay=WD,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    num_neighbors=NUM_NEIGHBORS,
)
refit_duration = time.time() - start_time
print(f"Refit Model Complete: Finished {best_epoch} epochs in {refit_duration:.1f}s")

# Save Trained Checkpoint
torch.save(hgt_refit_model.state_dict(), OUT_DIR / "hgt_model.pt")
print(f"Saved PyTorch checkpoint: {OUT_DIR / 'hgt_model.pt'}")

# Plot Learning Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="HGT Train Loss", color="#1f77b4", lw=1.8)
axes[0].set(xlabel="Epoch", ylabel="Loss", title="HGT Training Loss (Fit 1-24)")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["val_pr_auc"], label=f"Val PR-AUC (best={val_pr_auc_best:.4f})", color="#2ca02c", lw=1.8)
axes[1].axvline(best_epoch, color="#d62728", linestyle="--", alpha=0.7, label=f"Optimal Epoch ({best_epoch})")
axes[1].set(xlabel="Epoch", ylabel="PR-AUC", title="Validation PR-AUC (Window 25-34)")
axes[1].legend()
save_fig("learning_curves.png")


## 6. Operating Threshold Selection

Classification thresholds $\tau^*$ are chosen by **maximizing the F1 score strictly on the validation window (25–34)**.
The threshold $\tau^*$ is completely frozen before evaluating on any test period (35–49).

In [ ]:
def select_threshold(y_true: np.ndarray, scores: np.ndarray) -> tuple:
    '''Find F1-maximizing threshold on validation data strictly.'''
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype="float64")
    best_threshold, best_f1 = 0.5, -1.0
    for candidate in np.round(np.linspace(0.005, 0.995, 199), 4):
        value = f1_score(y_true, (scores >= candidate).astype(int), zero_division=0)
        if value > best_f1:
            best_threshold, best_f1 = float(candidate), float(value)
    return best_threshold, best_f1

val_mask = selection_data['tx'].val_mask
val_y = (txs.loc[val_mask.cpu().numpy(), "class"] == 1).values.astype(int)
val_probs = infer_hgt(hgt_selection_model, selection_data, val_mask, BATCH_SIZE, NUM_NEIGHBORS)

hgt_thr, hgt_val_f1 = select_threshold(val_y, val_probs)
print(f"HGT Operating Threshold : {hgt_thr:.3f} (Validation F1: {hgt_val_f1:.4f})")

check("threshold: HGT threshold selected on validation only", select_threshold(val_y, val_probs)[0], hgt_thr)

def _label_pos_weight(mask: np.ndarray) -> float:
    "pos_weight = negatives / positives over the masked supervision window."
    labels = tx_class[mask]
    return float((labels == 0).sum() / max(1, (labels == 1).sum()))

check(
    "loss: pos_weight (fit 1-24) derived from training labels only",
    round(float(pos_weight_fit.item()), 6),
    round(_label_pos_weight(is_labeled & (tx_step >= FIT_MIN) & (tx_step <= FIT_MAX)), 6),
)
check(
    "loss: pos_weight (train 1-34) derived from training labels only",
    round(float(pos_weight_train.item()), 6),
    round(_label_pos_weight(is_labeled & (tx_step >= TRAIN_MIN) & (tx_step <= TRAIN_MAX)), 6),
)
check("threshold: validation window is exactly steps 25-34", int(txs.loc[val_mask.cpu().numpy(), "Time step"].min()), VAL_MIN)
check("threshold: validation window upper bound is step 34", int(txs.loc[val_mask.cpu().numpy(), "Time step"].max()), VAL_MAX)


## 7. Primary Benchmark Evaluation (Test Period 35–49)

We evaluate HGT on the primary frozen test period (time steps 35–49; 16,670 labeled transactions; 1,083 illicit; 6.50% illicit prevalence).

We benchmark the new Heterogeneous Graph Transformer (HGT) against the complete progression of baselines:
1. **Prevalence Baseline**: Constant score equal to test prevalence (0.0650).
2. **Logistic Regression**: Linear tabular baseline (PR-AUC 0.2917).
3. **2-Layer MLP**: Feature-only neural baseline (PR-AUC 0.4768).
4. **GraphSAGE Baseline**: Homogeneous GNN baseline (PR-AUC 0.6216 / 0.6209).
5. **GraphSAGE Best Variant (O4)**: 3-layer depth control on the intra-step graph with weighted BCE (PR-AUC 0.6001).
6. **HeteroRGCN**: Multi-relational uniform mean aggregation GNN (PR-AUC 0.4682).
7. **XGBoost Baseline (500 Trees)**: Default tree-cap tabular baseline (PR-AUC 0.8007).
8. **XGBoost Optimized (Frozen)**: Retuned tabular ceiling (PR-AUC 0.8013).
9. **HeteroHGT**: Learned relation and neighborhood attention transformer.

In [ ]:
def evaluate(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    '''Compute all standard evaluation metrics at a fixed operating threshold.'''
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype="float64")
    predicted = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    prevalence = float(y_true.mean())
    pr_auc = float(average_precision_score(y_true, scores)) if y_true.sum() > 0 else float("nan")
    roc_auc = float(roc_auc_score(y_true, scores)) if 0 < y_true.sum() < y_true.size else float("nan")
    return {
        "n": int(y_true.size),
        "illicit": int(y_true.sum()),
        "prevalence": prevalence,
        "threshold": float(threshold),
        "pr_auc": pr_auc,
        "pr_auc_lift_over_prevalence": pr_auc / prevalence if prevalence else float("nan"),
        "roc_auc": roc_auc,
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "predicted_positives": int(predicted.sum()),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

tick("Generating test predictions on refit HGT model...")
test_mask = refit_data['tx'].test_mask
hgt_test_probs = infer_hgt(hgt_refit_model, refit_data, test_mask, BATCH_SIZE, NUM_NEIGHBORS)

y_test = (txs.loc[test_mask.cpu().numpy(), "class"] == 1).values.astype(int)
test_prevalence = float(y_test.mean())
eval_hgt = evaluate(y_test, hgt_test_probs, hgt_thr)

# Benchmark Progression
eval_prevalence = {
    "n": len(y_test), "illicit": int(y_test.sum()), "prevalence": test_prevalence, "threshold": test_prevalence,
    "pr_auc": test_prevalence, "pr_auc_lift_over_prevalence": 1.0, "roc_auc": 0.5000,
    "precision": test_prevalence, "recall": 1.0000, "f1": 2 * test_prevalence / (1 + test_prevalence),
    "predicted_positives": len(y_test), "tp": int(y_test.sum()), "fp": int(len(y_test) - y_test.sum()), "tn": 0, "fn": 0,
}
eval_logistic = {
    "n": len(y_test), "illicit": int(y_test.sum()), "prevalence": test_prevalence, "threshold": 0.655,
    "pr_auc": 0.2917, "pr_auc_lift_over_prevalence": 0.2917 / test_prevalence, "roc_auc": 0.8828,
    "precision": 0.3547, "recall": 0.5734, "f1": 0.4385, "predicted_positives": 1751,
    "tp": 621, "fp": 1130, "tn": 14457, "fn": 462,
}
eval_mlp = {
    "n": len(y_test), "illicit": int(y_test.sum()), "prevalence": test_prevalence, "threshold": 0.710,
    "pr_auc": 0.4768, "pr_auc_lift_over_prevalence": 0.4768 / test_prevalence, "roc_auc": 0.8912,
    "precision": 0.5964, "recall": 0.5568, "f1": 0.5759, "predicted_positives": 1011,
    "tp": 603, "fp": 408, "tn": 15179, "fn": 480,
}
eval_sage_base = {
    # GraphSAGE v2 Option 1 control, results/graphsage_v2/metrics.json (identical protocol to Phase 3).
    "n": len(y_test), "illicit": int(y_test.sum()), "prevalence": test_prevalence, "threshold": 0.820,
    "pr_auc": 0.6216, "pr_auc_lift_over_prevalence": 0.6216 / test_prevalence, "roc_auc": 0.9044,
    "precision": 0.6873, "recall": 0.5235, "f1": 0.5943, "predicted_positives": 825,
    "tp": 567, "fp": 258, "tn": 15329, "fn": 516,
}
eval_sage_o4 = {
    # GraphSAGE v2 Option 4 depth-3 control, results/graphsage_v2/metrics.json.
    "n": len(y_test), "illicit": int(y_test.sum()), "prevalence": test_prevalence, "threshold": 0.900,
    "pr_auc": 0.6001, "pr_auc_lift_over_prevalence": 0.6001 / test_prevalence, "roc_auc": 0.8923,
    "precision": 0.7427, "recall": 0.5143, "f1": 0.6077, "predicted_positives": 750,
    "tp": 557, "fp": 193, "tn": 15394, "fn": 526,
}
eval_rgcn = {
    "n": len(y_test), "illicit": int(y_test.sum()), "prevalence": test_prevalence, "threshold": 0.795,
    "pr_auc": 0.4682, "pr_auc_lift_over_prevalence": 0.4682 / test_prevalence, "roc_auc": 0.8946,
    "precision": 0.6241, "recall": 0.4598, "f1": 0.5295, "predicted_positives": 798,
    "tp": 498, "fp": 300, "tn": 15287, "fn": 585,
}
eval_xgb_base = {
    "n": len(y_test), "illicit": int(y_test.sum()), "prevalence": test_prevalence, "threshold": 0.515,
    "pr_auc": 0.8007, "pr_auc_lift_over_prevalence": 0.8007 / test_prevalence, "roc_auc": 0.9317,
    "precision": 0.8258, "recall": 0.7516, "f1": 0.7870, "predicted_positives": 986,
    "tp": 814, "fp": 172, "tn": 15415, "fn": 269,
}
eval_xgb_opt = {
    "n": len(y_test), "illicit": int(y_test.sum()), "prevalence": test_prevalence, "threshold": 0.435,
    "pr_auc": 0.8013, "pr_auc_lift_over_prevalence": 0.8013 / test_prevalence, "roc_auc": 0.9281,
    "precision": 0.7960, "recall": 0.7682, "f1": 0.7818, "predicted_positives": 1045,
    "tp": 832, "fp": 213, "tn": 15374, "fn": 251,
}

test_evaluations = {
    "Prevalence Baseline": eval_prevalence,
    "Logistic Regression": eval_logistic,
    "2-Layer MLP Baseline": eval_mlp,
    "GraphSAGE Baseline": eval_sage_base,
    "GraphSAGE Best Variant (O4)": eval_sage_o4,
    "HeteroRGCN": eval_rgcn,
    "XGBoost Baseline (500 Trees)": eval_xgb_base,
    "XGBoost Optimized (Frozen)": eval_xgb_opt,
    "HeteroHGT (Transformer)": eval_hgt,
}

comp_rows = []
for model_name, eval_dict in test_evaluations.items():
    comp_rows.append({
        "model": model_name,
        "pr_auc": eval_dict["pr_auc"],
        "roc_auc": eval_dict["roc_auc"],
        "precision": eval_dict["precision"],
        "recall": eval_dict["recall"],
        "f1": eval_dict["f1"],
        "tp": eval_dict["tp"],
        "fp": eval_dict["fp"],
        "fn": eval_dict["fn"],
        "threshold": eval_dict["threshold"],
    })

model_comparison_table = pd.DataFrame(comp_rows)
save_table("model_comparison.csv", model_comparison_table)
print("\n=== Model Benchmark Progression (Test 35-49) ===")
display(model_comparison_table.round(4))

# Plot Precision-Recall & ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# Precision-Recall Curve
p_hgt, r_hgt, _ = precision_recall_curve(y_test, hgt_test_probs)

axes[0].plot(r_hgt, p_hgt, label=f"HeteroHGT (PR-AUC={eval_hgt['pr_auc']:.4f})", color="#1f77b4", lw=2)
axes[0].axhline(0.8013, color="#d62728", linestyle=":", label="XGBoost Opt Ceiling (0.8013)")
axes[0].axhline(0.6216, color="#2ca02c", linestyle="--", label="GraphSAGE Baseline (0.6216)")
axes[0].axhline(0.4682, color="#9467bd", linestyle="-.", label="HeteroRGCN (0.4682)")
axes[0].axhline(test_prevalence, color="grey", linestyle="--", label=f"Prevalence ({test_prevalence:.4f})")
axes[0].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curve (Test 35-49)")
axes[0].legend(loc="upper right", fontsize=8)

# ROC Curve
order = np.argsort(hgt_test_probs)
y_sorted = y_test[order]
tpr = np.cumsum(y_sorted[::-1]) / y_sorted.sum()
fpr = np.cumsum((1 - y_sorted)[::-1]) / (1 - y_sorted).sum()
axes[1].plot(fpr, tpr, label=f"HeteroHGT (ROC-AUC={eval_hgt['roc_auc']:.4f})", color="#1f77b4", lw=1.8)
axes[1].plot([0, 1], [0, 1], color="grey", linestyle=":")
axes[1].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curve (Test 35-49)")
axes[1].legend(loc="lower right", fontsize=8)
save_fig("test_pr_roc.png")

check("evaluation: confusion matrix arithmetic closes for HGT", eval_hgt["tp"] + eval_hgt["fp"] + eval_hgt["tn"] + eval_hgt["fn"], len(y_test))


## 8. Temporal Sub-Window & Degradation Analysis (35–42 vs 43–49)

We evaluate HGT across three standardized temporal windows:
1. **Full Test Window (`35–49`)**: 16,670 labeled transactions, 1,083 illicit (6.50% prevalence).
2. **Early Test Window (`35–42` — Stationary Regime)**: 9,983 labeled transactions, 914 illicit (9.16% prevalence).
3. **Late Drift Window (`43–49` — Severe Shift Regime)**: 6,687 labeled transactions, 169 illicit (2.53% prevalence).

**The Scientific Test**:
Does HGT maintain predictive signal in the late drift window (`43–49`), or does it collapse to $\approx 0.05$ like XGBoost (0.0427), GraphSAGE (0.0505), and RGCN (0.0550)?

In [ ]:
test_step_vals = txs.loc[test_mask.cpu().numpy(), "Time step"].values

temporal_subwindows = {
    "test_full": ("35-49 (Primary)", np.ones(len(y_test), dtype=bool)),
    "test_early": ("35-42 (Early Test)", test_step_vals <= TEST_EARLY_MAX),
    "test_drift": ("43-49 (Recent Drift)", test_step_vals >= DRIFT_MIN),
}

temporal_rows = []
for win_key, (win_label, win_mask) in temporal_subwindows.items():
    y_win = y_test[win_mask]
    n_win = len(y_win)
    ill_win = int(y_win.sum())
    prev_win = float(y_win.mean())
    hgt_win_eval = evaluate(y_win, hgt_test_probs[win_mask], hgt_thr)

    # Benchmark baselines across windows
    xgb_pr_auc = 0.8013 if win_key == "test_full" else (0.9215 if win_key == "test_early" else 0.0427)
    sage_base_pr_auc = 0.6216 if win_key == "test_full" else (0.7352 if win_key == "test_early" else 0.0505)
    sage_o4_pr_auc = 0.6001 if win_key == "test_full" else (0.7596 if win_key == "test_early" else 0.0517)
    rgcn_pr_auc = 0.4682 if win_key == "test_full" else (0.6083 if win_key == "test_early" else 0.0550)

    temporal_rows.append({
        "window": win_label,
        "steps": {"test_full": "35-49", "test_early": "35-42", "test_drift": "43-49"}[win_key],
        "labeled": n_win,
        "illicit": ill_win,
        "prevalence": prev_win,
        "xgboost_pr_auc": xgb_pr_auc,
        "graphsage_base_pr_auc": sage_base_pr_auc,
        "graphsage_o4_pr_auc": sage_o4_pr_auc,
        "rgcn_pr_auc": rgcn_pr_auc,
        "hgt_pr_auc": hgt_win_eval["pr_auc"],
        "hgt_roc_auc": hgt_win_eval["roc_auc"],
        "hgt_f1": hgt_win_eval["f1"],
        "hgt_precision": hgt_win_eval["precision"],
        "hgt_recall": hgt_win_eval["recall"],
        "hgt_tp": hgt_win_eval["tp"],
        "hgt_fp": hgt_win_eval["fp"],
        "hgt_fn": hgt_win_eval["fn"],
        "hgt_tn": hgt_win_eval["tn"],
    })

temporal_table = pd.DataFrame(temporal_rows)
save_table("temporal_metrics.csv", temporal_table)
print("\n=== Temporal Sub-Window Metrics Across Architectures ===")
display(temporal_table.round(4))

# Plot Temporal Comparison Bar Chart
fig, ax = plt.subplots(figsize=(10, 4.8))
bar_width = 0.16
indices = np.arange(len(temporal_table))

ax.bar(indices - 2 * bar_width, temporal_table["xgboost_pr_auc"], bar_width, label="XGBoost Optimized", color="#d62728")
ax.bar(indices - bar_width, temporal_table["graphsage_base_pr_auc"], bar_width, label="GraphSAGE Baseline", color="#2ca02c")
ax.bar(indices, temporal_table["graphsage_o4_pr_auc"], bar_width, label="GraphSAGE O4", color="#17becf")
ax.bar(indices + bar_width, temporal_table["rgcn_pr_auc"], bar_width, label="HeteroRGCN", color="#9467bd")
ax.bar(indices + 2 * bar_width, temporal_table["hgt_pr_auc"], bar_width, label="HeteroHGT", color="#1f77b4")

for idx, prev in enumerate(temporal_table["prevalence"]):
    ax.plot([idx - 0.4, idx + 0.4], [prev, prev], color="grey", linestyle="--", linewidth=1.2)

ax.set_xticks(indices)
ax.set_xticklabels(temporal_table["steps"])
ax.set(xlabel="Time Step Sub-window", ylabel="PR-AUC", title="PR-AUC Across Temporal Windows (Dashed Line = Prevalence)")
ax.legend(fontsize=8, loc="upper right")
save_fig("temporal_pr_auc.png")

check("temporal: sub-windows partition labeled count", int(temporal_table.loc[temporal_table["steps"] != "35-49", "labeled"].sum()), int(temporal_table.loc[temporal_table["steps"] == "35-49", "labeled"].iloc[0]))
for _, _row in temporal_table.iterrows():
    check(
        f"confusion closure: HGT {_row['steps']}",
        int(_row["hgt_tp"] + _row["hgt_fp"] + _row["hgt_fn"] + _row["hgt_tn"]),
        int(_row["labeled"]),
    )


## 9. Inductive Evaluation: Seen vs. Unseen Address Context

To evaluate how well HGT generalizes to novel structural contexts:
1. **Seen Address Context**: Test transactions connected to at least one wallet address seen in historical training steps ($t \le 34$).
2. **Unseen Address Context**: Test transactions where all connected addresses first appear at step $t \ge 35$.
3. **No Address Linkage**: Singleton transactions with no connected wallet addresses (965 total nodes in graph).

In [ ]:
test_idx = np.where(test_mask.cpu().numpy())[0]

seen_mask = has_any_addr[test_idx] & (tx_min_addr_firstseen[test_idx] <= TRAIN_MAX)
unseen_mask = has_any_addr[test_idx] & (tx_min_addr_firstseen[test_idx] > TRAIN_MAX)
none_mask = ~has_any_addr[test_idx]

inductive_groups = [
    ("seen_address_context", seen_mask),
    ("unseen_address_context", unseen_mask),
    ("no_address_linkage", none_mask),
]

inductive_rows = []
for grp_name, grp_mask in inductive_groups:
    y_grp = y_test[grp_mask]
    if len(y_grp) > 0 and y_grp.sum() > 0:
        grp_eval = evaluate(y_grp, hgt_test_probs[grp_mask], hgt_thr)
        grp_eval["context"] = grp_name
        inductive_rows.append(grp_eval)
    elif len(y_grp) > 0:
        inductive_rows.append({
            "context": grp_name,
            "n": len(y_grp),
            "illicit": int(y_grp.sum()),
            "prevalence": 0.0,
            "pr_auc": float("nan"),
            "roc_auc": float("nan"),
            "precision": float("nan"),
            "recall": float("nan"),
            "f1": float("nan"),
            "tp": 0, "fp": 0, "tn": len(y_grp), "fn": 0,
        })

inductive_table = pd.DataFrame(inductive_rows)
save_table("inductive_metrics.csv", inductive_table)
print("\n=== Inductive Generalization Breakdown ===")
display(inductive_table.round(4))


## 10. Attention Weight Extraction & True-Positive Address Downweighting Test

### The Mechanistic Test
In Phase 4 and Phase 6, we identified that uniform mean aggregation over `AddrAddr` edges causes message dilution because high-degree exchange addresses dominate the neighborhood aggregation.

Here, we test whether HGT addresses this root cause:
1. **Layer-wise Relational Scalars (`a_rel`)**: We extract the learned scaling parameters for each of the 4 relations (`tx_to_tx`, `addr_to_tx`, `tx_to_addr`, `addr_to_addr`) across both HGT layers.
2. **True Positive Neighborhood Attention Inspection**:
   - We sample True Positives (`y_true == 1`, `score >= threshold`) from the test set.
   - We compute the edge-level attention scores $\alpha_{u \to v}$ for all incoming `AddrAddr` and `AddrTx` edges in their 2-hop subgraphs.
   - We correlate the assigned attention score with the **degree (reuse frequency)** of the source address node to test whether HGT actively suppresses high-degree hubs and prioritizes direct, low-degree wallet interactions.

In [ ]:
tick("Extracting relational attention scalars from trained HGT model...")

attn_scalar_records = []
for layer_idx, conv in enumerate(hgt_refit_model.convs):
    a_rel = getattr(conv, "a_rel", None)
    if a_rel is None:
        continue
    for rel_key, param in a_rel.items():
        rel_name = "__".join(rel_key) if isinstance(rel_key, tuple) else str(rel_key)
        if "__" in rel_name:
            rel_name = rel_name.split("__")[-2]
        mean_val = float(param.detach().cpu().mean().item())
        std_val = float(param.detach().cpu().std().item()) if param.numel() > 1 else 0.0
        attn_scalar_records.append({
            "layer": layer_idx + 1,
            "relation": rel_name,
            "mean_attention_scalar": mean_val,
            "std_attention_scalar": std_val,
        })

attn_scalar_df = pd.DataFrame(attn_scalar_records)
save_table("hgt_attention_weights.csv", attn_scalar_df)
print("\n=== Learned Relational Scaling Scalars (a_rel) ===")
display(attn_scalar_df.round(4))

tick("Analyzing edge-level attention scores on test True Positives...")
hgt_pred_test = (hgt_test_probs >= hgt_thr).astype(int)
tp_test_indices = np.where((y_test == 1) & (hgt_pred_test == 1))[0]
tp_tx_node_ids = test_idx[tp_test_indices]
print(f"Identified {len(tp_tx_node_ids)} test True Positives for attention inspection.")

# Sample up to 50 test True Positives for mechanistic attention inspection
sample_tp_txs = tp_tx_node_ids[:50]
tp_attn_df = pd.DataFrame()
attention_stats_df = pd.DataFrame()
attention_degree_corr_df = pd.DataFrame()

if len(sample_tp_txs) > 0:
    sample_loader = HeteroNeighborLoader(
        refit_data,
        num_neighbors=[15, 15],
        input_nodes=('tx', torch.tensor(sample_tp_txs, dtype=torch.long)),
        batch_size=len(sample_tp_txs),
        shuffle=False,
    )
    tp_batch = next(iter(sample_loader)).to(device)

    # Softmax-normalised attention weights, averaged over the 4 heads, for both address relations.
    try:
        addr_addr_weights, addr_addr_logits, addr_addr_edges = compute_hgt_edge_attention(
            hgt_refit_model, tp_batch.x_dict, tp_batch.edge_index_dict, ('addr', 'addr_to_addr', 'addr'), layer_idx=0
        )
        addr_tx_weights, addr_tx_logits, addr_tx_edges = compute_hgt_edge_attention(
            hgt_refit_model, tp_batch.x_dict, tp_batch.edge_index_dict, ('addr', 'addr_to_tx', 'tx'), layer_idx=0
        )
    except Exception as exc:
        # PyG occasionally renames HGTConv internals across releases; record the miss instead of aborting the run.
        print(f"Attention extraction unavailable with this PyG version ({type(exc).__name__}: {exc}).")
        addr_addr_weights = addr_addr_logits = np.array([])
        addr_tx_weights = addr_tx_logits = np.array([])
        addr_addr_edges = addr_tx_edges = np.empty((2, 0), dtype=np.int64)

    # Global address reuse frequency (in-degree over the full AddrAddr relation).
    addr_indegrees = np.bincount(dst_addr_addr, minlength=N_ADDR)
    addr_n_id = tp_batch['addr'].n_id.cpu().numpy() if hasattr(tp_batch['addr'], 'n_id') else np.arange(tp_batch['addr'].num_nodes)

    tp_attention_records = []
    for relation, weights, logits, edges in (
        ("addr_to_addr", addr_addr_weights, addr_addr_logits, addr_addr_edges),
        ("addr_to_tx", addr_tx_weights, addr_tx_logits, addr_tx_edges),
    ):
        for weight, logit, src_local in zip(weights, logits, edges[0]):
            global_src = int(addr_n_id[int(src_local)])
            deg = int(addr_indegrees[global_src])
            tp_attention_records.append({
                "relation": relation,
                "src_addr_idx": global_src,
                "src_addr_degree": deg,
                "attention_weight": float(weight),
                "attention_logit": float(logit),
                "degree_tier": "Hub (>50)" if deg > 50 else ("Medium (6-50)" if deg > 5 else "Low (1-5)"),
            })

    tp_attn_df = pd.DataFrame(tp_attention_records)
    save_table("tp_addr_attention.csv", tp_attn_df)

    if len(tp_attn_df) > 0:
        attention_stats_df = (
            tp_attn_df.groupby(["relation", "degree_tier"])["attention_weight"]
            .agg(["count", "mean", "std", "min", "max"])
            .reset_index()
        )
        save_table("attention_degree_stats.csv", attention_stats_df)
        print("\n=== Mean Attention Weight by Source Address Degree Tier ===")
        display(attention_stats_df.round(5))

    degree_corr_rows = []
    for relation in tp_attn_df["relation"].unique() if len(tp_attn_df) > 0 else []:
        sub = tp_attn_df[tp_attn_df["relation"] == relation]
        if sub["src_addr_degree"].nunique() > 1:
            corr = float(sub[["src_addr_degree", "attention_weight"]].corr(method="spearman").iloc[0, 1])
        else:
            corr = float("nan")
        degree_corr_rows.append({"relation": relation, "spearman_degree_vs_attention": corr, "n_edges": int(len(sub))})
    attention_degree_corr_df = pd.DataFrame(degree_corr_rows)
    if len(attention_degree_corr_df) > 0:
        save_table("attention_degree_correlation.csv", attention_degree_corr_df)
        display(attention_degree_corr_df.round(4))

    check("attention: sampled True Positives non-empty", int(len(sample_tp_txs) > 0), 1)
    check("attention: addr_to_addr edges captured in TP sample", int(len(addr_addr_weights)), None, note="informational; depends on the sampled ego-subgraph")
else:
    print("No test True Positives available for attention inspection; skipping.")
    check("attention: sampled True Positives non-empty", int(len(sample_tp_txs) > 0), 1)

# Plot Attention Diagnostics
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Relational Scalars Bar Plot
if len(attn_scalar_df) > 0:
    sns.barplot(data=attn_scalar_df, x="relation", y="mean_attention_scalar", hue="layer", ax=axes[0], palette="Blues_d")
    axes[0].tick_params(axis="x", rotation=20)
axes[0].set(xlabel="Relation Type", ylabel="Mean a_rel Parameter", title="Learned Relational Attention Scalars (a_rel)")

# Edge Attention vs Node Degree Scatter
if len(tp_attn_df) > 0:
    sns.scatterplot(
        data=tp_attn_df,
        x="src_addr_degree",
        y="attention_weight",
        hue="relation",
        alpha=0.6,
        ax=axes[1],
        palette=["#9467bd", "#ff7f0e"],
    )
    axes[1].set(xscale="log", xlabel="Source Address In-Degree (Log Scale)", ylabel="Softmax Attention Weight", title="True Positive Ego Subgraph: Attention vs Address Degree")
    axes[1].legend(loc="upper right")
else:
    axes[1].text(0.5, 0.5, "No sampled True Positives or edges", ha="center", va="center")
save_fig("attention_weights.png")


## 11. Test-Time Relation Ablation Comparison

To directly compare HGT's relational dependence against HeteroRGCN (which experienced severe performance shifts when relations were ablated in Phase 6), we ablate edge types at inference time:
1. **Full 4-Relation Graph**: `tx_to_tx`, `addr_to_tx`, `tx_to_addr`, `addr_to_addr`.
2. **Drop `addr_to_addr`**: Zeroing the 2.87M dense wallet-to-wallet transfer edges.
3. **Drop `tx_to_tx`**: Zeroing the 234,355 direct intra-step transaction edges.

In [ ]:
tick("Executing Test-Time Relation Ablation on HGT...")

ablation_configs = [
    ("Full Graph (4 Relations)", None),
    ("Drop addr_to_addr (No Wallet Transfers)", ('addr', 'addr_to_addr', 'addr')),
    ("Drop tx_to_tx (No Direct Tx Flow)", ('tx', 'tx_to_tx', 'tx')),
]

ablation_results = []
for config_name, drop_edge_type in ablation_configs:
    ablated_data = refit_data.clone()
    if drop_edge_type is not None:
        ablated_data[drop_edge_type].edge_index = torch.empty((2, 0), dtype=torch.long)

    ablated_probs = infer_hgt(hgt_refit_model, ablated_data, test_mask, BATCH_SIZE, NUM_NEIGHBORS)
    abl_eval = evaluate(y_test, ablated_probs, hgt_thr)
    ablation_results.append({
        "ablation": config_name,
        "dropped_relation": str(drop_edge_type) if drop_edge_type else "None",
        "pr_auc": abl_eval["pr_auc"],
        "roc_auc": abl_eval["roc_auc"],
        "precision": abl_eval["precision"],
        "recall": abl_eval["recall"],
        "f1": abl_eval["f1"],
        "pr_auc_delta": abl_eval["pr_auc"] - eval_hgt["pr_auc"],
    })

ablation_df = pd.DataFrame(ablation_results)
save_table("relation_ablation.csv", ablation_df)
print("\n=== Relation Ablation Results ===")
display(ablation_df.round(4))


## 12. Error Analysis & Temporal Diagnostics

We dissect the classification errors of HGT across time steps:
1. **Per-Step Confusion Matrix**: Tracking TP, FP, FN, TN, precision, recall, and F1 across each individual time step in $35 \le t \le 49$.
2. **Score Distribution**: Comparing predicted probability histograms for licit vs. illicit transactions.

In [ ]:
error_df = pd.DataFrame({
    "txId": tx_ids[test_idx],
    "time_step": test_step_vals,
    "y_true": y_test,
    "score": hgt_test_probs,
    "predicted": (hgt_test_probs >= hgt_thr).astype(int),
})

error_df["error_type"] = np.select(
    [
        (error_df.y_true == 1) & (error_df.predicted == 0),
        (error_df.y_true == 0) & (error_df.predicted == 1),
    ],
    ["FN", "FP"],
    default="correct",
)

per_step_errors = error_df.groupby("time_step").apply(
    lambda b: pd.Series({
        "n": len(b),
        "illicit": int(b.y_true.sum()),
        "tp": int(((b.y_true == 1) & (b.predicted == 1)).sum()),
        "fp": int(((b.y_true == 0) & (b.predicted == 1)).sum()),
        "fn": int(((b.y_true == 1) & (b.predicted == 0)).sum()),
        "tn": int(((b.y_true == 0) & (b.predicted == 0)).sum()),
        "precision": float(precision_score(b.y_true, b.predicted, zero_division=0)),
        "recall": float(recall_score(b.y_true, b.predicted, zero_division=0)),
        "f1": float(f1_score(b.y_true, b.predicted, zero_division=0)),
        "pr_auc": float(average_precision_score(b.y_true, b.score)) if b.y_true.sum() > 0 else float("nan"),
    })
).reset_index()
save_table("errors_by_step.csv", per_step_errors)

# Plot Error Trajectories
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(per_step_errors["time_step"], per_step_errors["fn"], marker="o", label="False Negatives (FN)", color="#d62728", lw=1.8)
axes[0].plot(per_step_errors["time_step"], per_step_errors["fp"], marker="s", label="False Positives (FP)", color="#ff7f0e", lw=1.8)
axes[0].axvline(DRIFT_MIN, color="grey", linestyle="--", alpha=0.7, label=f"Drift Boundary (t={DRIFT_MIN})")
axes[0].set(xlabel="Time Step", ylabel="Count", title="HGT Classification Errors per Time Step")
axes[0].legend()

axes[1].hist(error_df.loc[error_df.y_true == 1, "score"], bins=40, alpha=0.6, label="Illicit", color="#d62728")
axes[1].hist(error_df.loc[error_df.y_true == 0, "score"], bins=40, alpha=0.6, label="Licit", color="#1f77b4")
axes[1].axvline(hgt_thr, color="black", linestyle="--", label=f"Threshold ({hgt_thr:.3f})")
axes[1].set(xlabel="Predicted Score", ylabel="Count (Log Scale)", yscale="log", title="HGT Predicted Probability Distribution")
axes[1].legend()
save_fig("error_analysis.png")


In [ ]:
tick("Exporting full artifact suite to results/hgt/...")

# 1. Predictions CSV
predictions_df = pd.DataFrame({
    "txId": tx_ids[test_idx],
    "time_step": test_step_vals,
    "y_true": y_test,
    "hgt_score": hgt_test_probs,
    "hgt_predicted": (hgt_test_probs >= hgt_thr).astype(int),
})
save_table("predictions.csv", predictions_df)

# 2. Hyperparameters JSON
hyperparameters_payload = {
    "model": "Heterogeneous Graph Transformer (HGT)",
    "architecture": {
        "tx_in_channels": 165,
        "addr_in_channels": 55,
        "hidden_channels": 64,
        "out_channels": 1,
        "num_heads": 4,
        "num_layers": 2,
        "group_aggregation": "sum",
        "dropout": DROPOUT,
    },
    "training": {
        "batch_size": BATCH_SIZE,
        "num_neighbors": NUM_NEIGHBORS,
        "optimizer": "AdamW",
        "lr": LR,
        "weight_decay": WD,
        "loss": "BCEWithLogitsLoss",
        "pos_weight_fit": round(float(pos_weight_fit.item()), 4),
        "pos_weight_train": round(float(pos_weight_train.item()), 4),
        "best_epoch": best_epoch,
        "val_pr_auc_best": round(val_pr_auc_best, 4),
        "threshold": round(hgt_thr, 4),
    },
    "seed": SEED,
    "device": str(device),
}
save_json("hgt_hyperparameters.json", hyperparameters_payload)

# 3. Complete Metrics JSON
metrics_payload = {
    "protocol": {"fit": "1-24", "validation": "25-34", "train": "1-34", "test": "35-49"},
    "evaluations": test_evaluations,
    "temporal": temporal_table.to_dict(orient="records"),
    "inductive": inductive_table.to_dict(orient="records"),
    "ablation": ablation_df.to_dict(orient="records"),
    "attention_scalars": attn_scalar_df.to_dict(orient="records"),
    "attention_degree_stats": attention_stats_df.to_dict(orient="records"),
    "attention_degree_correlation": attention_degree_corr_df.to_dict(orient="records"),
    "hyperparameters": hyperparameters_payload,
}
save_json("metrics.json", metrics_payload)

# 3b. Dynamic conclusion (derived from measured metrics only — never pre-committed)
hgt_full_pr_auc = float(eval_hgt["pr_auc"])
hgt_drift_pr_auc = float(temporal_table.loc[temporal_table["steps"] == "43-49", "hgt_pr_auc"].iloc[0])
hgt_early_pr_auc = float(temporal_table.loc[temporal_table["steps"] == "35-42", "hgt_pr_auc"].iloc[0])
closes_aggregate_gap = hgt_full_pr_auc >= 0.75
recovers_drift_window = hgt_drift_pr_auc > 0.09
if len(attention_degree_corr_df) > 0:
    _attn_bits = [
        f"{_r.relation} Spearman {_r.spearman_degree_vs_attention:+.3f} (n={int(_r.n_edges)})"
        for _r in attention_degree_corr_df.itertuples()
    ]
    attention_note = "Attention weight vs source-address degree: " + "; ".join(_attn_bits)
else:
    attention_note = "Attention weight vs source-address degree: not computed (no sampled edges)."
conclusion = (
    f"HGT {'closes' if closes_aggregate_gap else 'does not close'} the aggregate gap to XGBoost "
    f"({hgt_full_pr_auc:.4f} vs 0.8013) and {'recovers' if recovers_drift_window else 'does not recover'} "
    f"the 43-49 drift window ({hgt_drift_pr_auc:.4f}; prior architectures all sit near 0.05)."
)

# 4. Text Digest
digest = f'''
================================================================================
BitcoinGraphGuard — Phase 7 Heterogeneous Graph Transformer (HGT) Digest
Generated: {time.strftime("%Y-%m-%d %H:%M:%S")}
================================================================================

1. Dataset & Graph Topology:
   - Transactions: 203,769 nodes (165 features)
   - Addresses: 822,942 nodes (55 features)
   - Relations: tx->tx (234k), addr->tx (477k), tx->addr (837k), addr->addr (2.87M)
   - Protocol: Fit 1-24 | Val 25-34 | Train 1-34 | Test 35-49

2. Benchmark Progression on Primary Test (35-49):
   - Prevalence Baseline          : PR-AUC {eval_prevalence['pr_auc']:.4f}
   - Logistic Regression          : PR-AUC {eval_logistic['pr_auc']:.4f} | F1 {eval_logistic['f1']:.4f}
   - 2-Layer MLP Baseline         : PR-AUC {eval_mlp['pr_auc']:.4f} | F1 {eval_mlp['f1']:.4f}
   - HeteroRGCN (Mean Agg)        : PR-AUC {eval_rgcn['pr_auc']:.4f} | F1 {eval_rgcn['f1']:.4f}
   - GraphSAGE Baseline           : PR-AUC {eval_sage_base['pr_auc']:.4f} | F1 {eval_sage_base['f1']:.4f}
   - GraphSAGE O4 (Cross-Step/Lag): PR-AUC {eval_sage_o4['pr_auc']:.4f} | F1 {eval_sage_o4['f1']:.4f}
   - XGBoost Optimized (Ceiling)  : PR-AUC {eval_xgb_opt['pr_auc']:.4f} | F1 {eval_xgb_opt['f1']:.4f}
   - HeteroHGT (Transformer)      : PR-AUC {eval_hgt['pr_auc']:.4f} | F1 {eval_hgt['f1']:.4f}

3. Temporal Sub-Window Breakdown:
   - Window 35-42 (Stationary) : HGT PR-AUC = {temporal_table.loc[temporal_table['steps']=='35-42', 'hgt_pr_auc'].iloc[0]:.4f} | ROC-AUC = {temporal_table.loc[temporal_table['steps']=='35-42', 'hgt_roc_auc'].iloc[0]:.4f} (vs XGB 0.9215 | SAGE 0.7352 | RGCN 0.6083)
   - Window 43-49 (Late Drift) : HGT PR-AUC = {temporal_table.loc[temporal_table['steps']=='43-49', 'hgt_pr_auc'].iloc[0]:.4f} | ROC-AUC = {temporal_table.loc[temporal_table['steps']=='43-49', 'hgt_roc_auc'].iloc[0]:.4f} (vs XGB 0.0427 | SAGE 0.0505 | RGCN 0.0550)

4. Key Scientific Conclusion:
   - {conclusion}
   - {attention_note}
================================================================================
'''
print(digest)
(OUT_DIR / "digest.txt").write_text(digest, encoding="utf-8")

# 5. Automated Checks
check("predictions: HGT binary prediction matches score >= threshold", int((predictions_df["hgt_predicted"] != (predictions_df["hgt_score"] >= hgt_thr).astype(int)).sum()), 0)
check("checks: test count equals 16670", len(predictions_df), 16670)

save_table("checks.csv", pd.DataFrame(CHECKS))
failures = [r for r in CHECKS if not r["ok"]]
print(f"\nAutomated Assertion Suite: {len(CHECKS)} checks completed, {len(failures)} failures.")
if failures:
    display(pd.DataFrame(failures))


## 13. Final Verdict & Phase 8 (MLOps) Decision

The verdict below is **rendered from the measured run** — its text and `verdict.json` are derived
from `eval_hgt`, the three temporal windows, and the true-positive attention sample, not written in
advance. The Phase 8 decision rule is fixed here and applied to whatever the run produced:

* **HGT becomes the frozen serving model** only if it (a) clears the simple GNN baselines on 35–49
  and lands within striking distance of XGBoost, **and** (b) escapes the ~0.05 band on 43–49.
* Otherwise **XGBoost Optimized remains the frozen model** and HGT is at most an auxiliary
  structural explainer. XGBoost's aggregate PR-AUC (0.8013) is 0.18–0.33 above every GNN while it
  shares the same 43–49 weakness, so freezing a weaker model buys nothing.

In [ ]:
tick("Rendering the Phase 7 verdict from the measured artifacts...")
from IPython.display import Markdown, display

drift_row = temporal_table.loc[temporal_table["steps"] == "43-49"].iloc[0]
early_row = temporal_table.loc[temporal_table["steps"] == "35-42"].iloc[0]

XGBOOST_PR_AUC = 0.8013
SAGE_BASELINE = 0.6216
SAGE_O4 = 0.6001
RGCN = 0.4682
DRIFT_FLOOR = 0.09  # 43-49 band where every prior frozen architecture sits (~0.04-0.055)

hgt_full = float(eval_hgt["pr_auc"])
hgt_early = float(early_row["hgt_pr_auc"])
hgt_drift = float(drift_row["hgt_pr_auc"])
gap_to_xgboost = XGBOOST_PR_AUC - hgt_full

beats_rgcn = bool(hgt_full > RGCN)
beats_sage_baseline = bool(hgt_full > SAGE_BASELINE)
beats_sage_o4 = bool(hgt_full > SAGE_O4)
closes_aggregate_gap = bool(hgt_full >= 0.75)
recovers_drift_window = bool(hgt_drift > DRIFT_FLOOR)

# Mechanistic test: did attention down-weight high-degree reused addresses on AddrAddr edges?
addr_addr_attn = tp_attn_df[tp_attn_df["relation"] == "addr_to_addr"] if len(tp_attn_df) > 0 else pd.DataFrame()
hub_minus_low = None
degree_spearman = float("nan")
if len(addr_addr_attn) > 0:
    hubs = addr_addr_attn.loc[addr_addr_attn["src_addr_degree"] > 50, "attention_weight"]
    lows = addr_addr_attn.loc[addr_addr_attn["src_addr_degree"] <= 5, "attention_weight"]
    if len(hubs) > 0 and len(lows) > 0:
        hub_minus_low = float(hubs.mean() - lows.mean())
    if addr_addr_attn["src_addr_degree"].nunique() > 1:
        degree_spearman = float(addr_addr_attn[["src_addr_degree", "attention_weight"]].corr(method="spearman").iloc[0, 1])
attention_downweights_hubs = bool(hub_minus_low is not None and hub_minus_low < 0)

verdict_payload = {
    "protocol": {"fit": "1-24", "validation": "25-34", "train": "1-34", "test": "35-49"},
    "hgt_pr_auc_35_49": hgt_full,
    "hgt_pr_auc_35_42": hgt_early,
    "hgt_pr_auc_43_49": hgt_drift,
    "xgboost_pr_auc_35_49": XGBOOST_PR_AUC,
    "gap_to_xgboost_35_49": gap_to_xgboost,
    "beats_rgcn_35_49": beats_rgcn,
    "beats_graphsage_baseline_35_49": beats_sage_baseline,
    "beats_graphsage_o4_35_49": beats_sage_o4,
    "closes_aggregate_gap_35_49": closes_aggregate_gap,
    "recovers_drift_window_43_49": recovers_drift_window,
    "attention_downweights_hubs": attention_downweights_hubs,
    "addr_addr_hub_minus_low_attention": hub_minus_low,
    "addr_addr_degree_spearman": degree_spearman,
    "frozen_model_recommendation": "HGT" if (closes_aggregate_gap and recovers_drift_window) else "XGBoost",
}
save_json("verdict.json", verdict_payload)

# --- Verdict text, branch selected by the measured numbers ---
if recovers_drift_window:
    drift_headline = "**Verdict on 43-49: attention partially closes the gap.**"
    drift_verdict = (
        f"HGT moves the 43-49 window to PR-AUC {hgt_drift:.4f}, against the ~0.04-0.055 band "
        f"(XGBoost 0.0427, GraphSAGE 0.0505, RGCN 0.0550). Learned attention changes the drift regime."
    )
elif hgt_drift >= 0.07:
    drift_headline = "**Verdict on 43-49: attention moves the number but does not close the gap.**"
    drift_verdict = (
        f"HGT lands at {hgt_drift:.4f} on 43-49, nominally above the 0.04-0.055 band but still far "
        f"below any usable operating point. Attention does not rescue the drift window."
    )
else:
    drift_headline = "**Verdict on 43-49: attention does not close the gap.**"
    drift_verdict = (
        f"HGT collapses to {hgt_drift:.4f} on 43-49, inside the identical ~0.05 band as XGBoost "
        f"(0.0427), GraphSAGE (0.0505) and RGCN (0.0550). Replacing mean aggregation with learned "
        f"multi-head attention changes nothing in the drift regime."
    )

if attention_downweights_hubs:
    attention_verdict = (
        f"On AddrAddr edges inside sampled true-positive ego-subgraphs, high-degree hubs (>50) "
        f"receive {hub_minus_low:+.5f} attention weight relative to low-degree addresses (Spearman "
        f"{degree_spearman:+.3f}). HGT does learn to down-weight reused high-volume addresses, so the "
        f"Phase 4/6 dilution mechanism is real and attention is the right lever for it - yet the "
        f"aggregate and drift scores show that fixing dilution is not sufficient on its own."
    )
elif hub_minus_low is not None:
    attention_verdict = (
        f"HGT did not down-weight hubs: high-degree addresses (>50) receive {hub_minus_low:+.5f} "
        f"attention weight relative to low-degree addresses (Spearman {degree_spearman:+.3f}). The "
        f"dilution problem is not being corrected at the attention layer."
    )
else:
    attention_verdict = (
        "No AddrAddr edges were captured in the true-positive sample, so the mechanistic attention "
        "test is inconclusive this run."
    )

if closes_aggregate_gap and recovers_drift_window:
    production_line = "Freeze **HGT** as the Phase 8 serving model."
elif hgt_full > SAGE_BASELINE and not recovers_drift_window:
    production_line = (
        "HGT is the strongest GNN on 35-49 but is still more than 0.18 PR-AUC below XGBoost and "
        "equally blind on 43-49. Freeze **XGBoost Optimized** and keep HGT only as an auxiliary "
        "structural explainer."
    )
else:
    production_line = (
        "HGT does not beat the simpler GNN baselines and does not touch the drift window. Freeze "
        "**XGBoost Optimized** as the serving model."
    )

verdict_md = f'''## 13. Final Verdict (measured)

### 1. Does attention close the aggregate gap on 35-49?
**HGT PR-AUC = {hgt_full:.4f}** (ROC-AUC {eval_hgt['roc_auc']:.4f}, F1 {eval_hgt['f1']:.4f}) against
XGBoost {XGBOOST_PR_AUC:.4f}, GraphSAGE baseline {SAGE_BASELINE:.4f}, GraphSAGE O4 {SAGE_O4:.4f} and
RGCN {RGCN:.4f}. Beats RGCN: {beats_rgcn}. Beats GraphSAGE baseline: {beats_sage_baseline}. Beats
GraphSAGE O4: {beats_sage_o4}. Remaining gap to XGBoost: {gap_to_xgboost:+.4f}. On the stationary
early window 35-42 HGT reaches PR-AUC {hgt_early:.4f} (XGBoost 0.9215).

### 2. Does attention close the 43-49 gap?
{drift_headline} {drift_verdict}

### 3. Did HGT learn to down-weight high-degree reused addresses (the Phase 4/6 mechanism)?
{attention_verdict}

### 4. What this means for Phase 8 (MLOps)
{production_line}

XGBoost's aggregate PR-AUC (0.8013) is 0.18-0.33 above every GNN, and no architecture tested - RGCN,
GraphSAGE with cross-step edges and lag features, or HGT with learned attention - has produced signal
in 43-49. The evidence therefore no longer points to an aggregation-weighting defect as the dominant
cause of the late-window collapse: it points to an **out-of-distribution regime shift** (new entities
and patterns genuinely absent from 1-34; adversarial validation AUC 1.0000, Phase 2b). No frozen
architecture - attention or otherwise - repairs that; only label refresh, continuous retraining, or
explicit drift detection can, which is exactly the Phase 8 mandate.
'''

(OUT_DIR / "verdict.md").write_text(verdict_md, encoding="utf-8")
display(Markdown(verdict_md))

# Package the artifact set so the Colab/Kaggle run can be pulled back to the repository root.
try:
    import zipfile

    archive = Path("hgt_artifacts.zip")
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
        for _file in sorted(OUT_DIR.rglob("*")):
            if _file.is_file():
                zf.write(_file, _file.relative_to(OUT_DIR.parent))
    print(f"Artifact archive ready: {archive.resolve()}")
except Exception as _exc:
    print(f"Artifact archive skipped: {_exc}")

tick("Phase 7 verdict written to verdict.json / verdict.md and rendered above.")
